# 🏁 Racing Gym: teach a car to race, in 10 minutes, in your browser

You will:
1. look at the world through the car's sensors,
2. measure two baselines (random driving, and a hand-written controller),
3. **train a neural network with reinforcement learning** (PPO, ~2 minutes),
4. watch it go from crashing to racing,
5. download your agent and **race it in the [live playground](https://alperen012.github.io/Reinforcement_learning_car/web/)**.

No GPU needed. `Runtime → Run all` works. Project: [https://github.com/Alperen012/Reinforcement_learning_car](https://github.com/Alperen012/Reinforcement_learning_car)

In [ ]:
%pip install -q "racing-gym @ git+https://github.com/Alperen012/Reinforcement_learning_car"

## 1 · The environment

The **agent** (the driver) and the **environment** (track, physics, rules) take turns:
the agent sends an **action**, the environment answers with an **observation** and a **reward**.

In [ ]:
import matplotlib.pyplot as plt
from racing_gym import RaceEnv, EnvConfig

env = RaceEnv(EnvConfig(track="oval", random_start=False), render_mode="rgb_array")
obs, info = env.reset(seed=0)
print("action space:", env.action_space, "  -> [steer, throttle]")
print("observation: 17 numbers")
for name, value in zip(env.obs_names, obs):
    print(f"  {name:<16} {value:+.2f}")
plt.figure(figsize=(9, 5)); plt.imshow(env.render()); plt.axis("off"); plt.show()

That is **all** the agent ever sees: no picture, no map, no x/y position. The lidar rays
(drawn on the image) measure the distance to the track edge.

Now take a few random actions and look at the **reward**. It is a sum of terms, and the
environment tells you each one:

In [ ]:
env.action_space.seed(0)
for step in range(5):
    obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
    terms = {k: round(v, 3) for k, v in info["reward_terms"].items() if v}
    print(f"step {step}: reward {reward:+.3f} = {terms}")

## 2 · Baselines: what does "good" even mean?

Before training anything, measure two yardsticks on the **same 5 seeded episodes**:
- **random**: random steering and throttle,
- **heuristic**: three lines of hand-written control (steer towards the track direction).

In [ ]:
from racing_gym.evaluation import heuristic_policy, random_policy, run_episodes, summarize

cfg = EnvConfig(track="oval")
probe = RaceEnv(cfg)
for name, policy in [("random", random_policy(probe)), ("heuristic", heuristic_policy(probe))]:
    s = summarize(run_episodes(policy, cfg, episodes=5))
    print(f"{name:<10} return {s['return_mean']:8.1f}   laps {s['laps_mean']:.1f}   crash {s['crash_rate']:.0%}   speed {s['avg_speed']:.2f}")

## 3 · Train an agent with reinforcement learning

PPO starts from a random neural network and only ever sees observations and rewards.
Every 10,000 steps it is tested on 3 episodes. Watch `crash` fall and `laps` rise.
(~2 minutes on a Colab CPU.)

In [ ]:
from racing_gym.config import load_config
from racing_gym.training import train

exp = load_config(None, [
    "name=my_first_agent", "algo=ppo", "env.track=oval",
    "train.total_timesteps=80000", "train.n_envs=8", "train.eval_freq=10000", "train.eval_episodes=3",
])
run_dir = train(exp, seed=0, out_dir="runs")

In [ ]:
import csv
rows = list(csv.DictReader(open(run_dir / "evals.csv")))
t = [int(r["timesteps"]) for r in rows]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(t, [float(r["return_mean"]) for r in rows], "o-"); ax[0].set_title("evaluation return")
ax[1].plot(t, [float(r["crash_rate"]) for r in rows], "o-", color="tab:red"); ax[1].set_title("crash rate")
for a in ax: a.set_xlabel("environment steps"); a.grid(alpha=.3)
plt.show()

## 4 · Before and after

The same network, untrained (left) and trained (right), on the same track:

In [ ]:
from IPython.display import Image, display
from racing_gym.gif import record_gif

gif = record_gif([str(run_dir / "checkpoints" / "step_000000000.zip"), str(run_dir)],
                 EnvConfig(track="oval", random_start=False), "before_after.gif", steps=400, every=3, scale=0.45)
display(Image(filename=str(gif)))

Does it generalise? Try it on a track it has **never seen**: (Spoiler: often not. Why? See tutorial 5.)

In [ ]:
from racing_gym.evaluation import make_policy
policy, agent_cfg, label = make_policy(str(run_dir), EnvConfig(track="hairpin"))
for track in ["oval", "hairpin", "random:3"]:
    s = summarize(run_episodes(policy, agent_cfg, [track], episodes=3))
    print(f"{track:<10} laps {s['laps_mean']:.1f}   crash {s['crash_rate']:.0%}")

## 5 · Race your agent in the browser

Export it, download the file, then open the **[playground](https://alperen012.github.io/Reinforcement_learning_car/web/)** and drop the file
on the track. Your car will race the project's agents, and you can scrub through *its own*
training with the stage slider.

In [ ]:
from racing_gym.export import export_policy
out = export_policy(run_dir, "my_agent.js", name="my_first_agent", stages=5)
print("wrote", out)
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print("(not on Colab: the file is next to this notebook)")

## 6 · Your turn: experiments

Change **one thing** in the `load_config` cell above and re-run from there:

| try | what you will learn |
|---|---|
| `"env.reward.crash=0"` | does the agent still avoid walls? (crashing still ends the episode…) |
| `"env.reward.center=-0.5"` | the reward decides the driving line |
| `"env.car=rocket"` | a faster car with less grip: harder to learn? |
| `"algo=sac"` (and `train.total_timesteps=40000`) | off-policy learning: fewer steps, more compute per step |
| `"env.track=random"` | a new track every episode: does it generalise now? |

Next: the tutorials in [docs/tutorials](https://github.com/Alperen012/Reinforcement_learning_car/tree/main/docs/tutorials), the
[challenges](https://github.com/Alperen012/Reinforcement_learning_car/blob/main/CHALLENGES.md) (can you beat the leaderboard?), or ⭐ the repo if it helped.